In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types  import *
import sys
sys.path.append("/Workspace/Users/sivana9908_gmail.com#ext#@sivana9908gmail.onmicrosoft.com/Uber-Eats-End-to-End-_Azure-Data-Engineering-Project")
from src.common.spark_utils import standardize_columns
from delta.tables import DeltaTable

In [0]:
#Reading data from adls 
df_promotions= (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("abfss://bronze@ubereaststorage.dfs.core.windows.net/csv/promotions/")
)
source_rows = df_promotions.count()
print(f"promotions.source_rows={source_rows}")
display(df_promotions)
df_promotions.printSchema()

In [0]:
# standardize the columns
df_promotions=standardize_columns(df_promotions)


                       
#deduplication 
df_promotions = df_promotions.drop_duplicates(["promo_id","restaurant_id"]) 

#applying business rules
df_promotions = df_promotions.filter(col("discount_value")>0)

display(df_promotions)










In [0]:
try:
    table_name = "ubereats_databricks1.silver.promotions"
    
    if not spark.catalog.tableExists(table_name):
    
        df_promotions.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(table_name)
    
    else:
    
        target = DeltaTable.forName(spark, table_name)
    
        (
            target.alias("t")
            .merge(
                df_promotions.alias("s"),
                "t.promo_id = s.promo_id AND t.restaurant_id = s.restaurant_id",
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
except Exception as exc:
    # Record publish failures without replacing the original exception.
    failure_record = [{
        "run_id": str(__import__("uuid").uuid4()),
        "dataset_name": "promotions",
        "stage": "silver_publish",
        "error_type": type(exc).__name__,
        "error_message": str(exc)[:2000],
        "recorded_at_utc": str(spark.sql("SELECT current_timestamp() AS ts").first()["ts"]),
    }]
    try:
        spark.createDataFrame(failure_record).write.format("delta").mode("append").saveAsTable(
            "ubereats_databricks1.silver.pipeline_failure_events"
        )
    except Exception as log_exc:
        print("promotions.failure_log_error=" + str(log_exc)[:500])
    raise

# Persist a compact successful-run quality record for later monitoring.
quality_table = "ubereats_databricks1.silver.pipeline_quality_results"
quality_record = [{
    "run_id": str(__import__("uuid").uuid4()),
    "dataset_name": "promotions",
    "source_rows": int(source_rows),
    "prepared_rows": int(df_promotions.count()),
    "target_rows_after": int(spark.table(table_name).count()),
    "status": "SUCCEEDED",
    "recorded_at_utc": str(spark.sql("SELECT current_timestamp() AS ts").first()["ts"]),
}]
spark.createDataFrame(quality_record).write.format("delta").mode("append").saveAsTable(quality_table)
print("promotions.quality.persisted_run_id=" + quality_record[0]["run_id"])
